# 문제 20. 이탈 위험 스코어카드

recency·frequency·최근 30일 방문여부 3요소를 점수화해 위험점수(0~6)를 만들고,
과거 가치(상반기 순매출)와 교차해 '고위험 x 고가치' 우선 관리군을 뽑습니다.
외부 모듈 없이, `data/orders.csv`, `data/order_items.csv`, `data/web_logs.csv`만으로 이 노트북 하나로 완결됩니다.

## 0. 환경 설정

In [11]:
import pandas as pd

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
REF_DATE = pd.Timestamp("2024-07-01")   # 관측 기준일
RECENCY_CUTOFF = 45                       # 문제19 이탈선(60일)보다 앞선 조기경보용 컷


---
## 1. 유효 주문 + 매출(라인) 데이터 준비

1장 기준 그대로: 중복 제거 -> 날짜 파싱 -> 2024 상반기 필터 -> canceled/returned 제외 -> line_amount 파생

In [10]:
orders = pd.read_csv("data/orders.csv").drop_duplicates()
items = pd.read_csv("data/order_items.csv").drop_duplicates()

orders["order_datetime"] = pd.to_datetime(orders["order_datetime"], errors="coerce")
period_mask = (orders["order_datetime"] >= "2024-01-01") & (orders["order_datetime"] < "2024-07-01")
orders = orders[period_mask].copy()

net_lines = orders.merge(items, on="order_id", how="inner").dropna(subset=["unit_price"])
net_lines = net_lines[~net_lines["status"].isin(["canceled", "returned"])].copy()
net_lines["line_amount"] = net_lines["quantity"] * net_lines["unit_price"] * (1 - net_lines["discount"])

print(f"순매출 라인 수: {len(net_lines):,}건 / 고유 고객 수: {net_lines['customer_id'].nunique():,}명")
net_lines.head()


순매출 라인 수: 407,686건 / 고유 고객 수: 5,714명


,order_id,customer_id,order_datetime,channel,status,order_item_id,product_id,quantity,unit_price,discount,line_amount
0,25463,1077,2024-06-25 00:17:41,store,delivered,265800,104,5,"73,300.00",0.40,"219,900.00"
1,25463,1077,2024-06-25 00:17:41,store,delivered,16468,319,4,"68,800.00",0.21,"217,408.00"
2,25463,1077,2024-06-25 00:17:41,store,delivered,477391,9,4,"19,400.00",0.34,"51,216.00"
3,171088,2998,2024-05-28 19:35:20,web,delivered,55193,279,4,"28,000.00",0.02,"109,760.00"
4,171088,2998,2024-05-28 19:35:20,web,delivered,200065,3,2,"20,300.00",0.32,"27,608.00"


## 2. 고객별 recency · frequency · value 계산

- **recency**: 마지막 주문 후 경과일 (날짜 단위로 통일해 계산)
- **frequency**: 상반기 유효 주문 건수
- **value**: 상반기 순매출 합계

In [3]:
order_level = net_lines.groupby(["customer_id", "order_id"]).agg(
    order_datetime=("order_datetime", "first"),
    order_amount=("line_amount", "sum"),
).reset_index()

last_purchase = order_level.groupby("customer_id")["order_datetime"].max().dt.normalize()
recency = (REF_DATE.normalize() - last_purchase).dt.days.rename("recency")

frequency = order_level.groupby("customer_id")["order_id"].nunique().rename("frequency")
value = order_level.groupby("customer_id")["order_amount"].sum().rename("value")

score_df = pd.concat([recency, frequency, value], axis=1)
print(f"고객 수: {len(score_df):,}명")
score_df.head()


고객 수: 5,714명


,recency,frequency,value
customer_id,,,
1000,11,7,"2,530,766.00"
1001,8,22,"5,849,515.00"
1002,16,9,"2,432,546.00"
1003,2,56,"18,760,198.00"
1004,2,23,"7,727,622.00"


---
## 3. web_logs에서 6월(최근 30일) 방문 고객 집합 만들기

100만 행이므로 `usecols`로 필요한 두 열만, `dtype`으로 customer_id 결측을 안전하게 읽습니다.

In [ ]:
logs = pd.read_csv(
    "data/web_logs.csv",
    usecols=["customer_id", "event_time"],
    dtype={"customer_id": "float64"},
    parse_dates=["event_time"],
)

june_visitors = set(
    logs.loc[logs["event_time"] >= "2024-06-01", "customer_id"].dropna().unique()
)

print(f"6월 방문 고객 수: {len(june_visitors):,}명 (로그에 없는 고객은 미방문으로 처리)")


6월 방문 고객 수: 4,945명 (로그에 없는 고객은 미방문으로 처리)


---
## 4. 3요소 점수화 (0~2점씩) -> 위험점수(0~6)

점수는 '좋은 신호'일수록 높게 매깁니다 (최근 구매·잦은 구매·최근 방문 = 안전).
그래서 마지막에 `6 - 위험점수`로 뒤집어야 '높을수록 진짜 이탈 위험'이 되는 점수가 됩니다.

In [5]:
freq_q = score_df["frequency"].quantile([0.33, 0.66])

def r_score(r):
    return 2 if r <= RECENCY_CUTOFF * 0.5 else (1 if r <= RECENCY_CUTOFF else 0)

def f_score(f):
    return 2 if f >= freq_q[0.66] else (1 if f >= freq_q[0.33] else 0)

score_df["R점수"] = score_df["recency"].apply(r_score)
score_df["F점수"] = score_df["frequency"].apply(f_score)
score_df["방문점수"] = score_df.index.to_series().apply(lambda c: 2 if c in june_visitors else 0)

score_df["안전점수"] = score_df["R점수"] + score_df["F점수"] + score_df["방문점수"]
score_df["이탈위험점수"] = 6 - score_df["안전점수"]

score_df[["recency", "frequency", "R점수", "F점수", "방문점수", "이탈위험점수"]].head()


,recency,frequency,R점수,F점수,방문점수,이탈위험점수
customer_id,,,,,,
1000,11,7,2,0,2,2
1001,8,22,2,2,2,0
1002,16,9,2,1,2,1
1003,2,56,2,2,2,0
1004,2,23,2,2,2,0


### 스코어 규칙표

| 요소 | 2점 | 1점 | 0점 |
|---|---|---|---|
| recency | 컷(45일)의 절반, 약 22일 이내 | 45일 이내 | 그보다 오래됨 |
| frequency | 상위 34%(P66 이상) | 중간 33%(P33~P66) | 하위 33% |
| 6월 방문 | 방문함(2점) | - | 미방문(0점) |

**이탈위험점수 = 6 - (R점수+F점수+방문점수)**, 0~6점, 높을수록 위험.


---
## 5. 위험 x 가치 분포

이탈위험점수가 4점 이상이면 고위험, value가 중앙값 이상이면 고가치로 정의합니다.

In [6]:
value_median = score_df["value"].median()
score_df["고가치"] = score_df["value"] >= value_median
score_df["고위험"] = score_df["이탈위험점수"] >= 4

print(f"가치 중앙값: {value_median:,.0f}원\n")
print("[위험 x 가치 분포]")
pd.crosstab(score_df["고위험"], score_df["고가치"])


가치 중앙값: 3,592,020원

[위험 x 가치 분포]


고가치,False,True
고위험,,
False,1979,2855
True,878,2


---
## 6. 우선 관리군(고위험 x 고가치) 목록

In [7]:
priority = score_df[score_df["고위험"] & score_df["고가치"]].sort_values("value", ascending=False)

print(f"우선 관리군: {len(priority):,}명")
priority[["recency", "frequency", "value", "이탈위험점수"]].head(10)


우선 관리군: 2명


,recency,frequency,value,이탈위험점수
customer_id,,,,
4248,57,8,"3,900,832.00",4
4537,24,9,"3,595,831.00",4


## 7. 예상 매출 손실 추정 (연 환산 가정)

**가정**: 상반기(6개월) 순매출을 2배 해서 연간으로 환산합니다.
실제 미래 구매 예측이 아니라, '지금까지의 페이스가 유지된다면'이라는 단순 가정입니다.

In [8]:
annualized_loss = priority["value"].sum() * 2

print(f"우선 관리군 {len(priority):,}명 전원 이탈 시,")
print(f"연 환산(상반기 x 2) 예상 손실: 약 {annualized_loss:,.0f}원")


우선 관리군 2명 전원 이탈 시,
연 환산(상반기 x 2) 예상 손실: 약 14,993,326원
